# Предсказание стоимости жилья

В проекте вам нужно обучить модель линейной регрессии на данных о жилье в Калифорнии в 1990 году. На основе данных нужно предсказать медианную стоимость дома в жилом массиве. Обучите модель и сделайте предсказания на тестовой выборке. Для оценки качества модели используйте метрики RMSE, MAE и R2.

In [1]:
import pandas as pd 
import numpy as np

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.types import *
import pyspark.sql.functions as F

from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler, StandardScaler, Imputer
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

In [2]:
RANDOM_SEED = 52

## Загрузка данных

Для начала инициализируем сессию Spark

In [3]:
# Инициализируем локальную Spark сессию
spark = SparkSession.builder \
                    .master("local") \
                    .appName("CaliforniaHousingLinearRegression") \
                    .getOrCreate()

# Проверяем, что сессия создана
print("Spark session created successfully!")
print(f"Spark version: {spark.version}")
print(f"Spark app name: {spark.sparkContext.appName}")

Spark session created successfully!
Spark version: 3.0.2
Spark app name: CaliforniaHousingLinearRegression


Загрузим данные и проверим корректность загрузки

In [4]:
# Читаем данные
df = spark.read.option('header', 'true').csv('/datasets/housing.csv', inferSchema = True)

In [5]:
# Проверяем типы колонок
df.printSchema()

root
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- housing_median_age: double (nullable = true)
 |-- total_rooms: double (nullable = true)
 |-- total_bedrooms: double (nullable = true)
 |-- population: double (nullable = true)
 |-- households: double (nullable = true)
 |-- median_income: double (nullable = true)
 |-- median_house_value: double (nullable = true)
 |-- ocean_proximity: string (nullable = true)



Данные загрузились корректно. Все колонки кроме `ocean_proximity` - числовые.

## Подготовка данных

Начнём подготовку данных. Для этого нам нужно сделать следующие шаги:
 - Выявить и заполнить пропуски (если таковые имеются)
 - Закодировать категориальную колонку (будем использовать OHE)
 - Отмасштабировать значения числовых признаков (будем использовать StandardScaler)
 
Всю предобработку и саму модель положим в пайплайн

Заранее запомним, какие признаки категориальные, какие числовые, какой у нас таргет

In [6]:
categorical_cols = ['ocean_proximity']
numerical_cols = ['median_income', 'households', 'population', 'total_bedrooms', 'total_rooms', 
                 'housing_median_age', 'latitude', 'longitude']
target = 'median_house_value'

Посмотрим на пропуски в данных

In [7]:
# Выведем описательные статистики для каждой колонки
df.describe().toPandas()

,summary,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,count,20640,20640,20640,20640,20433,20640,20640,20640,20640,20640
1,mean,-119.56970445736148,35.6318614341087,28.639486434108527,2635.7630813953488,537.8705525375618,1425.4767441860465,499.5396802325581,3.8706710029070246,206855.81690891474,None
2,stddev,2.003531723502584,2.135952397457101,12.58555761211163,2181.6152515827944,421.38507007403115,1132.46212176534,382.3297528316098,1.899821717945263,115395.61587441359,None
3,min,-124.35,32.54,1.0,2.0,1.0,3.0,1.0,0.4999,14999.0,<1H OCEAN
4,max,-114.31,41.95,52.0,39320.0,6445.0,35682.0,6082.0,15.0001,500001.0,NEAR OCEAN


Так как нам нужно будет обучать модели на разных наборах данных, то и есть смысл создать 2 пайплайна. Для создания пайплайна напишем функцию с флагом: используем мы категориальную фичу или нет.    

Шаги пайплайна:
 - Заполняем пропуски в колонке с пропусками
 - Приводим числовые признаки к виду, который понимают алгоритмы
 - Масштабируем числовые признаки
 - Если используем категориальный признак:
  - Приводим его к виду, понятному для алгоритмов
  - Кодируем (OHE)
  - Собираем все признаки в вектор
 - Иначе:
  - Собираем в вектор только числовые признаки
 - Добавляем модель в пайплайн

In [8]:
def create_pipeline(use_categorical=True):
    stages = []
    
    # 1. Imputer для пропусков
    stages.append(
        Imputer(
            inputCols=["total_bedrooms"],
            outputCols=["total_bedrooms"],
            strategy="median"
        )
    )
    
    # 2. Сборка числовых признаков
    stages.append(
        VectorAssembler(
            inputCols=numerical_cols,
            outputCol="numerical_features_raw"
        )
    )
    
    # 3. Масштабирование числовых признаков
    stages.append(
        StandardScaler(
            inputCol="numerical_features_raw",
            outputCol="numerical_features_scaled",
            withStd=True,
            withMean=True
        )
    )
    
    # 4. Если нужно добавить категориальные признаки
    if use_categorical:
        # Индексация категорий
        stages.append(
            StringIndexer(
                inputCols=categorical_cols,
                outputCols=[c + '_idx' for c in categorical_cols],
                handleInvalid="keep"
            )
        )
        
        # OneHot кодирование
        stages.append(
            OneHotEncoder(
                inputCols=[c + '_idx' for c in categorical_cols],
                outputCols=[c + '_ohe' for c in categorical_cols],
                dropLast=False
            )
        )
        
        # Сборка категориальных признаков
        stages.append(
            VectorAssembler(
                inputCols=[c + '_ohe' for c in categorical_cols],
                outputCol="categorical_features"
            )
        )
        
        # Финальная сборка всех признаков
        stages.append(
            VectorAssembler(
                inputCols=["numerical_features_scaled", "categorical_features"],
                outputCol="features"
            )
        )
    else:
        # Используем только масштабированные числовые признаки
        stages.append(
            VectorAssembler(
                inputCols=["numerical_features_scaled"],
                outputCol="features"
            )
        )
    
    # 5. Модель линейной регрессии
    stages.append(
        LinearRegression(
            labelCol=target,
            featuresCol="features",
            predictionCol="prediction"
        )
    )
    
    return Pipeline(stages=stages)

# Обучение моделей

Начнём обучение с разбиение на тренировочную и тестовую выборки в соотношение 80 на 20 соответственно

In [9]:
# Разбиваем данные
train_data, test_data = df.randomSplit([.8,.2], seed=RANDOM_SEED)
print(train_data.count(), test_data.count()) 

16603 4037


Создадим 2 пайплайна. Для числовых и всех фичей. Обучим модели и сделаем предсказания

In [10]:
# Создаём пайплайн
pipeline_numerical = create_pipeline(use_categorical=False)
# Обучаем модель (включает всю предобработку)
model_numerical = pipeline_numerical.fit(train_data)
# Делаем предсказания
predictions_numerical = model_numerical.transform(test_data)
predictions_numerical.select(target, 'prediction').show(3)

26/03/27 15:02:29 WARN Instrumentation: [3c7eb15e] regParam is zero, which might cause numerical instability and overfitting.
26/03/27 15:02:30 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeSystemBLAS
26/03/27 15:02:30 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeRefBLAS
26/03/27 15:02:30 WARN LAPACK: Failed to load implementation from: com.github.fommil.netlib.NativeSystemLAPACK
26/03/27 15:02:30 WARN LAPACK: Failed to load implementation from: com.github.fommil.netlib.NativeRefLAPACK


+------------------+-----------------+
|median_house_value|       prediction|
+------------------+-----------------+
|           50800.0|183940.7005380594|
|           66900.0|67469.32230110446|
|           68400.0|78785.52504616298|
+------------------+-----------------+
only showing top 3 rows



In [11]:
# Создаём пайплайн
pipeline_all = create_pipeline(use_categorical=True)
# Обучаем модель (включает всю предобработку)
model_all = pipeline_all.fit(train_data)
# Делаем предсказания
predictions_all = model_all.transform(test_data)
predictions_all.select(target, 'prediction').show(3)

26/03/27 15:02:36 WARN Instrumentation: [763d2a78] regParam is zero, which might cause numerical instability and overfitting.
26/03/27 15:02:37 WARN Instrumentation: [763d2a78] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.


+------------------+------------------+
|median_house_value|        prediction|
+------------------+------------------+
|           50800.0|212656.65792071592|
|           66900.0|115527.62854581414|
|           68400.0|126954.55095747698|
+------------------+------------------+
only showing top 3 rows



Получили предсказания обеих моделей. Перейдём к анализу результатов

# Анализ результатов

В этой секции будем сравнивать результаты двух моделей по метрикам RMSE, MAE и R2. Для этого напишем функцию

In [12]:
def evaluate_model(predictions):
    # Создаем evaluator для разных метрик
    # RMSE
    evaluator_rmse = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName="rmse")
    rmse = evaluator_rmse.evaluate(predictions)

    # MAE
    evaluator_mae = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName="mae")
    mae = evaluator_mae.evaluate(predictions)

    # R2
    evaluator_r2 = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName="r2")
    r2 = evaluator_r2.evaluate(predictions)

    print("=== Model Evaluation Metrics ===")
    print(f"RMSE: {rmse:.2f}")
    print(f"MAE: {mae:.2f}")
    print(f"R²: {r2:.4f}")

Применим функцию для каждых предсказаний

In [13]:
evaluate_model(predictions_numerical)

=== Model Evaluation Metrics ===
RMSE: 69283.48
MAE: 50404.57
R²: 0.6300


In [14]:
evaluate_model(predictions_all)

=== Model Evaluation Metrics ===
RMSE: 68022.32
MAE: 49078.70
R²: 0.6433


Не забываем закрыть сессию

In [15]:
spark.stop()

Модель в которой участвовали все признаки справилась немного лучше чем модель, которая обучалась только на числовых признаках. Это весьма логично, т.к. первая модель не брала во внимание важный признак удаления домовладения от океана `ocean_proximity` что по ощущениям неплохо влияет на медианную ценность дома. Конечно для подтверждения этой гипотезы лучше провести анализ данных, но в рамках данного проекта ограничимся этим